# 02 — Noise and filters: implement them yourself

**Goal:** understand when to use moving-average low-pass, exponential low-pass, median, and difference high-pass filters, and implement each using ordinary Python.

Python packages for DSP are **not** required. The optional plotting cell uses `matplotlib` if installed. The decoder cells use only `random`, `sum`, `len`, `sorted`, slicing, loops, and arithmetic.

## Filtering intuition

- **Moving average (low-pass FIR):** average neighboring samples to suppress rapid random fluctuations. But it blurs edges and introduces delay when causal.
- **Exponential moving average (EMA, low-pass IIR):** `y[n] = alpha*x[n] + (1-alpha)*y[n-1]`. Small `alpha` is smoother but slower.
- **Median filter:** replaces a point with the middle ranked value in a neighborhood; excellent for isolated spikes.
- **Difference (simple high-pass):** `y[n] = x[n] - x[n-1]`. Highlights fast changes, but amplifies high-frequency noise; it does not directly recover a clean data level.

**Watch for:** sampling rate, edge delay, and filter-window size relative to symbol duration.

In [ ]:
import random
rng = random.Random(42)
truth = [0, 0, 1, 1, 0, 1, 0, 1]
samples_per_bit = 12
samples = []
for bit in truth:
    for _ in range(samples_per_bit):
        noisy_value = bit + rng.gauss(0, 0.23)
        samples.append(noisy_value)
samples[34] += 2.2   # a single spike to test median filtering
print('First 12 samples:', [round(x, 2) for x in samples[:12]])

## Exercise 1 — Moving average, causal low-pass

For each index `i`, average up to the most recent `width` samples, including `samples[i]`. At the start of the signal, use however many samples exist.

C/C++ loop translation: `for i in range(len(samples))`, then `start = max(0, i-width+1)` and `window = samples[start:i+1]`.

In [ ]:
def moving_average(values, width):
    result = []
    for i in range (len(values)):
        start = max (0,i-width+1)
        window = values[start:i+1]
        result.append (sum(window)/len(window))
    return result
    # TODO: append the average of each trailing window
    raise NotImplementedError

assert moving_average([1, 2, 3, 4], 2) == [1.0, 1.5, 2.5, 3.5]
print('Low-pass test passed')

Low-pass test passed


## Exercise 2 — EMA, first-order IIR low-pass

Start at `values[0]`. For each subsequent sample, calculate `filtered = alpha*current + (1-alpha)*previous_filtered`.

Try `alpha = 0.2` and `alpha = 0.8`. Which responds faster when the bit changes?

In [10]:
def ema(values, alpha):
    # TODO: return [] for empty input; otherwise return one output per input
    
    if (len(values) == 0):
        return []
    
    result = []
    result.append(values[0])
    for i in range (1, len(values)):
        filtered = (alpha*values[i]) + ((1-alpha) * result[i-1])
        result.append(filtered)
    return result
    raise NotImplementedError

assert ema([], 0.5) == []
assert ema([0, 1, 1], 0.5) == [0, 0.5, 0.75]
print('EMA test passed')

EMA test passed


## Exercise 3 — Median filter for spikes

Use odd-width windows centered around the current index, with clipped boundaries. `sorted(window)[len(window)//2]` gets the middle ranked element of each window. At boundaries, clipped windows can be even; for this exercise, take the upper middle item.

**Question:** why might a median be preferable to a mean when a single sample jumps to 10 V?

In [23]:
def median_filter(values, width):
    #if even 
    filtered = []

    for i in range (len(values)):
        start = max(0,i-width+1)
        window = sorted(values[start:i+1])
        #even
        filtered.append(window[(len(window)//2)])

    
    # TODO: require a positive odd width, sort each centered window
    return filtered
    raise NotImplementedError
    

assert median_filter([0, 0, 9, 0, 0], 3) == [0, 0, 0, 0, 0]
print('Median test passed')

Median test passed


## Exercise 4 — A simple high-pass / edge detector

First-order difference: `output[0] = 0`, `output[i] = values[i] - values[i-1]` for `i >= 1`. This highlights edges and noise. Do not use it alone to decide whether a digital level is high or low.

In [26]:
def difference(values):
    # TODO: return [] if empty; output starts at 0 for nonempty
    if (len(values) == 0):return []
    difference = []
    difference.append(0)
    for i in range (1,len(values)):
        difference.append(values[i] - values[i-1])
    return difference
    raise NotImplementedError

assert difference([0, 0, 1, 1, 0]) == [0, 0, 1, 0, -1]
print('High-pass test passed')

High-pass test passed


## Compare the outputs (optional plotting)

Only this cell needs `matplotlib`. If it's not installed, filter implementations and tests above still work. Plot the original and filtered signals to spot noise removal and delayed transitions.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print('Optional: pip install matplotlib to display plots')
else:
    plt.figure(figsize=(11, 4))
    plt.plot(samples, alpha=0.35, label='Noisy samples')
    plt.plot(moving_average(samples, 5), label='Moving average width=5')
    plt.plot(ema(samples, 0.25), label='EMA alpha=0.25')
    plt.plot(median_filter(samples, 5), label='Median width=5')
    plt.axhline(0.5, linestyle='--', color='gray', label='Threshold=0.5')
    plt.title('Filtering a noisy digital waveform')
    plt.xlabel('Sample index')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.show()

## Thinking questions (answer in your own words)

1. If one bit lasts 12 samples, why might a 24-sample moving average be a bad idea?
2. What happens to a threshold crossing when `alpha` becomes very small?
3. Which filter is best for occasional large spikes? Which is most convenient for steady random noise?
4. Is a moving-average output aligned with the original transitions, or does it lag?
5. Would you low-pass the raw high-frequency carrier of an OOK signal directly, or first compute an envelope/energy measure?

**Next:** 03 — UART frame recovery and 04 — OOK demodulation.